# Taxonomy Aware ESM Training Submission

This notebook submits the training job to Azure ML.
It uses `src/train.py` and the dataset `azureml:MHD:2`.

In [6]:
from azure.ai.ml import MLClient, command, Input
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml.entities import Environment
import logging
import os

# Decrease logging noise for Azure Identity
logging.getLogger("azure.identity").setLevel(logging.WARNING)

# 1. Connect to Azure ML
print("Connecting to Azure ML...")

# STRICT CONFIGURATION
TENANT_ID = "3569a77b-5841-417b-b138-12b5fe698c9c"

# We enforce InteractiveBrowserCredential to ensure the correct tenant login.
try:
    credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)
    # Force a token fetch to trigger the login prompt immediately
    credential.get_token("https://management.azure.com/.default")
    
    ml_client = MLClient.from_config(credential=credential)
    print(f"Connected to workspace: {ml_client.workspace_name}")
except Exception as e:
    print(f"Authentication failed: {e}")
    raise e

Connecting to Azure ML...


Found the config file in: .\config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Connected to workspace: azure_ML


In [7]:
# --- Environment Configuration ---
USE_EXIST_ENV = True  # Set to False to rebuild environment
ENV_NAME = "cafa6-torch-env"
ENV_VERSION = "5"      # Increment version
ENV_PATH = "C:/workspace/CAFA_6_hjs/env"

if USE_EXIST_ENV:
    print(f"Using existing environment: {ENV_NAME}:{ENV_VERSION}")
    env = f"{ENV_NAME}:{ENV_VERSION}"
else:
    print(f"Building new environment: {ENV_NAME}:{ENV_VERSION} from {ENV_PATH}")
    env = Environment(
        name=ENV_NAME,
        version=ENV_VERSION,
        image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
        conda_file=f"{ENV_PATH}/conda.yaml",
        description="Environment with PyTorch, ESM, and PEFT for LoRA"
    )
    ml_client.environments.create_or_update(env)
    print("Environment built and registered.")


Using existing environment: cafa6-torch-env:5


## Parameter Center
Adjust the training hyperparameters below.

In [8]:
# --- Training Hyperparameters ---
ESM_MODEL = "facebook/esm2_t6_8M_UR50D" 
LEARNING_RATE = 1e-4
MIN_LR = 1e-6
BATCH_SIZE = 64
EPOCHS = 20
NUM_WORKERS = 8

# --- Scheduler Settings (CosineAnnealingWarmRestarts) ---
T_0 = 10 
T_MULT = 1

# --- Asymmetric Loss Parameters ---
GAMMA_NEG = 4
GAMMA_POS = 0
CLIP = 0.05

# --- LoRA Parameters ---
USE_LORA = True
LORA_RANK = 8

# --- Command Construction ---
training_command = (
    f"python train.py "
    f"--data_path ${{inputs.dataset}} "
    f"--epochs {EPOCHS} "
    f"--batch_size {BATCH_SIZE} "
    f"--lr {LEARNING_RATE} "
    f"--min_lr {MIN_LR} "
    f"--num_workers {NUM_WORKERS} "
    f"--T_0 {T_0} "
    f"--T_mult {T_MULT} "
    f"--esm_model_name {ESM_MODEL} "
    f"--gamma_neg {GAMMA_NEG} "
    f"--gamma_pos {GAMMA_POS} "
    f"--clip {CLIP} "
    f"--use_lora {USE_LORA} "
    f"--lora_rank {LORA_RANK}"
)

print(f"Training Command: {training_command}")

Training Command: python train.py --data_path ${inputs.dataset} --epochs 20 --batch_size 64 --lr 0.0001 --min_lr 1e-06 --num_workers 8 --T_0 10 --T_mult 1 --esm_model_name facebook/esm2_t6_8M_UR50D --gamma_neg 4 --gamma_pos 0 --clip 0.05 --use_lora True --lora_rank 8


In [9]:
# 2. Configure Job

job = command(
    code="./src",
    command=training_command,
    inputs={
        "dataset": Input(
            type="uri_folder",
            path="azureml:cafa6:3"
        )
    },
    environment=f"{ENV_NAME}:{ENV_VERSION}", 
    compute="FOR-CAFA-6",
    display_name="TaxonomyAwareESM_LoRA",
    experiment_name="taxonomy_aware_esm"
)


In [10]:
# 3. Submit Job
returned_job = ml_client.create_or_update(job)
print(f"Job submitted. Studio URL: {returned_job.studio_url}")

Uploading src (9.53 MBs): 100%|##########| 9533825/9533825 [00:02<00:00, 3421392.45it/s] 


Use of {} for parameters is deprecated, instead use ${{}}.


Job submitted. Studio URL: https://ml.azure.com/runs/stoic_bone_p2s7v1g7zg?wsid=/subscriptions/c39e1d7a-fa06-4185-94ce-18b724cac3d8/resourcegroups/js_p_j/workspaces/azure_ML&tid=3569a77b-5841-417b-b138-12b5fe698c9c
